In [11]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"D:\DS115-VIS\.venv\Project1_Predictive-Maintenance\ai4i_fused.csv")
df.head()


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,PWF,OSF,RNF,date,avg_temp_c,min_temp_c,max_temp_c,precipitation_mm,avg_wind_speed_kmh,avg_sea_level_pres_hpa
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4


In [12]:
# Clean trailing and leading spaces from column names
df.columns = df.columns.str.strip()

In [13]:
# 2. Sort data for time-series compatibility
# Chronological order is mandatory for accurate rolling calculations
df = df.sort_values(by=['date', 'UDI']).reset_index(drop=True)

In [14]:
# 3. Select Target Telemetry Features
# These are the main continuous sensor and weather columns from image
sensor_features = [
    'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
    'avg_temp_c', 'min_temp_c', 'max_temp_c', 'precipitation_mm', 'aavg_wind_speed_kmh', 'avg_sea_level_pres_hpa'
]

In [15]:
# Define the rolling window size (Operational window size - e.g., 5 logs)
window_size = 5
df.head(5)

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,PWF,OSF,RNF,date,avg_temp_c,min_temp_c,max_temp_c,precipitation_mm,avg_wind_speed_kmh,avg_sea_level_pres_hpa
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4


In [16]:
print("Starting feature engineering...")


Starting feature engineering...


In [17]:
# 4. Engineer Rolling Mean, Standard Deviation, and Variance
# Perform calculations inside a new DataFrame
engineered_features = pd.DataFrame(index=df.index)


In [19]:
for col in sensor_features:
    # Check if the column exists in the dataset
    if col in df.columns:
        # Rolling Mean
        engineered_features[f'{col}_roll_mean'] = df[col].rolling(window=window_size).mean()
        
        # Rolling Standard Deviation
        engineered_features[f'{col}_roll_std'] = df[col].rolling(window=window_size).std()
        
        # Rolling Variance
        engineered_features[f'{col}_roll_var'] = df[col].rolling(window=window_size).var()

In [20]:
# 5. Combine with original metadata and targets
# Includes UDI, Product ID, Machine failure, and specific failure modes
meta_cols = ['UDI', 'Product ID', 'Type', 'date', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
# Filter out names to match what is present in the dataset
meta_cols = [c for c in meta_cols if c in df.columns]

final_df = pd.concat([df[meta_cols], engineered_features], axis=1)


In [22]:
# 6. Drop NaN rows generated by the rolling window warmup buffer
final_df.dropna(inplace=True)
df.head(5)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,PWF,OSF,RNF,date,avg_temp_c,min_temp_c,max_temp_c,precipitation_mm,avg_wind_speed_kmh,avg_sea_level_pres_hpa
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4


In [23]:
# 7. Verify the output structure and save the file
print(f"Processed dataset shape: {final_df.shape}")
print(final_df.head())

# Save the final engineered dataset
final_df.to_csv("engineered_iot_weather_dataset.csv", index=False)
print("File successfully saved as 'engineered_iot_weather_dataset.csv'!")


Processed dataset shape: (9996, 40)
   UDI Product ID Type        date  Machine failure  TWF  HDF  PWF  OSF  RNF  \
4    5     L47184    L  2020-01-01                0    0    0    0    0    0   
5    6     M14865    M  2020-01-01                0    0    0    0    0    0   
6    7     L47186    L  2020-01-01                0    0    0    0    0    0   
7    8     L47187    L  2020-01-01                0    0    0    0    0    0   
8    9     M14868    M  2020-01-01                0    0    0    0    0    0   

   ...  min_temp_c_roll_var  max_temp_c_roll_mean  max_temp_c_roll_std  \
4  ...                  0.0                  30.3                  0.0   
5  ...                  0.0                  30.3                  0.0   
6  ...                  0.0                  30.3                  0.0   
7  ...                  0.0                  30.3                  0.0   
8  ...                  0.0                  30.3                  0.0   

   max_temp_c_roll_var  precipitation_

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [25]:
# 1. Define original sensor features & target variable
# Using only the original features as requested for the baseline
base_features = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
target = 'Machine failure'  
# Assuming 'Machine failure' stands for Machine Failure 
# Filter features that exist in df to avoid any KeyErrors
base_features = [col for col in base_features if col in df.columns]

In [26]:
# 2. Prepare the dataset
# Drop any rows where features or target might have missing values
baseline_df = df[base_features + [target]].dropna()

X = baseline_df[base_features]
y = baseline_df[target]

In [27]:
# 3. Train-Test Split (Stratified to handle class imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [28]:
from imblearn.over_sampling import SMOTE

In [29]:
# 4. Initialize SMOTE and resample the training data
print("Applying SMOTE and Training Random Forest...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Train using the resampled data
baseline_rf = RandomForestClassifier(random_state=42, n_estimators=100)
baseline_rf.fit(X_train_resampled, y_train_resampled)

# 5. Make Predictions (Keep testing on original X_test)
y_pred = baseline_rf.predict(X_test)


# 6. Evaluate and print scores
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='binary') # Use 'macro' if it is multi-class

print("\n================ BASELINE MODEL METRICS ================")
print(f"Accuracy Score : {accuracy:.4f}")
print(f"F1-Score       : {f1:.4f}")
print("========================================================")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Applying SMOTE and Training Random Forest...

================ BASELINE MODEL METRICS ================
Accuracy Score : 0.9580
F1-Score       : 0.5579

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.96      0.98      1932
           1       0.43      0.78      0.56        68

    accuracy                           0.96      2000
   macro avg       0.71      0.87      0.77      2000
weighted avg       0.97      0.96      0.96      2000



In [30]:
# 1. Apply SMOTE and Train
print("Applying SMOTE + Probability Tuning...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

baseline_rf = RandomForestClassifier(random_state=42, n_estimators=100)
baseline_rf.fit(X_train_resampled, y_train_resampled)

# 2. Use predict_proba instead of predict
y_probs = baseline_rf.predict_proba(X_test)[:, 1]

# 3. Tweak this value (Try 0.6 or 0.7 to raise precision)
custom_threshold = 0.65  
y_pred = (y_probs >= custom_threshold).astype(int)

# Evaluate and print scores
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='binary')

print("\n================ NEW TUNED MODEL METRICS ================")
print(f"Accuracy Score : {accuracy:.4f}")
print(f"F1-Score       : {f1:.4f}")
print("========================================================")
print("\nClassification Report:\n", classification_report(y_test, y_pred))



Applying SMOTE + Probability Tuning...

================ NEW TUNED MODEL METRICS ================
Accuracy Score : 0.9695
F1-Score       : 0.6211

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.98      0.98      1932
           1       0.54      0.74      0.62        68

    accuracy                           0.97      2000
   macro avg       0.76      0.86      0.80      2000
weighted avg       0.98      0.97      0.97      2000



In [31]:
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score

In [32]:
# 1. Import Advanced Boosting Algorithms
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

models = {
    'XGBoost': XGBClassifier(),
    'LightGBM': LGBMClassifier(),
    'CatBoost': CatBoostClassifier()
}

print("Initializing Advanced Model Exploration Framework...")

# 2. Define the Pipelines (SMOTE + Model)
# Idhu tuning pannum podhu data leak aagama automation-ah SMOTE panni train panni paakum.
pipelines = {
    'XGBoost': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0))
    ]),
    'LightGBM': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', LGBMClassifier(random_state=42, verbose=-1))
    ]),
    'CatBoost': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', CatBoostClassifier(random_state=42, verbose=0))
    ])
}

Initializing Advanced Model Exploration Framework...


In [33]:
# 3. Define Hyperparameter Tuning Grids
# Hyperparameters-ah limit panni set panyiruken, appo dhaan GridSearchCV seekiram run aagum.
param_grids = {
    'XGBoost': {
        'model__n_estimators': [0.05, 0.1],
        'model__max_depth': [0.05, 0.1],
        'model__learning_rate': [0.05, 0.1]
    },
    'LightGBM': {
        'model__n_estimators': [0.05, 0.1],
        'model__max_depth': [0.05, 0.1],
        'model__learning_rate': [0.05, 0.1]
    },
    'CatBoost': {
        'model__iterations': [0.05, 0.1],
        'model__depth': [0.05, 0.1],
        'model__learning_rate': [0.05, 0.1]
    }
}


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, classification_report

# 4. Dictionary to store best models
best_models = {}

# 1. Pipelines definition
# Handle imbalanced data by setting class_weight='balanced' to give more weight to minority class failures
pipelines = {
    'lr': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ]),
    'rf': Pipeline([
        ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
    ])
}

# 2. Param Grids definition
# Specified appropriate estimator counts for RandomForest to avoid non-finite or invalid grid search fits
param_grids = {
    'lr': {
        'classifier__C': [0.01, 0.1, 1, 10]
    },
    'rf': {
        'classifier__n_estimators': [None, 10, 20],  
        'classifier__max_depth': [None, 10, 20]
    }
}

# 3. Run GridSearchCV for each model
for name in pipelines.keys():
    print(f"\nTuning hyperparameters for {name} using GridSearchCV...")
    
    grid_search = GridSearchCV(
        estimator=pipelines[name],
        param_grid=param_grids[name],
        cv=3,                 # 3-Fold Cross Validation
        scoring='f1',         # Optimize based on F1-Score
        n_jobs=-1,            # Use all available CPU cores for speed
        verbose=1
    )
    # Train the Grid Search
    grid_search.fit(X_train, y_train)
    
    # Save the best estimator
    best_models[name] = grid_search.best_estimator_
    print(f"Best Params for {name}: {grid_search.best_params_}")

# 4. Evaluate All Tuned Advanced Models on Test Data
print("\n================ ADVANCED MODELS EVALUATION ================")
for name, model in best_models.items():
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='binary')
    
    print(f"\n--- {name} Performance ---")
    print(f"Optimized F1-Score: {f1:.4f}")
    print(classification_report(y_test, y_pred))



Tuning hyperparameters for lr using GridSearchCV...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best Params for lr: {'classifier__C': 0.1}

Tuning hyperparameters for rf using GridSearchCV...
Fitting 3 folds for each of 9 candidates, totalling 27 fits


d:\DS115-VIS\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
9 fits failed out of a total of 27.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
9 fits failed with the following error:
Traceback (most recent call last):
  File "d:\DS115-VIS\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\DS115-VIS\.venv\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "d:\DS115-VIS\.venv\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params

Best Params for rf: {'classifier__max_depth': 10, 'classifier__n_estimators': 20}

================ ADVANCED MODELS EVALUATION ================

--- lr Performance ---
Optimized F1-Score: 0.2395
              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1932
           1       0.14      0.84      0.24        68

    accuracy                           0.82      2000
   macro avg       0.57      0.83      0.57      2000
weighted avg       0.96      0.82      0.87      2000


--- rf Performance ---
Optimized F1-Score: 0.6345
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      1932
           1       0.60      0.68      0.63        68

    accuracy                           0.97      2000
   macro avg       0.79      0.83      0.81      2000
weighted avg       0.98      0.97      0.97      2000

